# 結合した 2 量子ビット

このノートブックでは、結合した 2 つのトランズモン量子ビットの常在 ZZ 相互作用を調べる方法を示します。

## 背景

2 つの量子ビットが結合していると、それぞれの遷移周波数は独立ではなくなり、相手の状態に依存した周波数シフトが生じます。
トランズモンでは、結合強度 $g$、周波数差 $\Delta = \omega_0 - \omega_1$、非調和度 $\alpha_0, \alpha_1$ によって、分散的な相互作用があらわれます。

このノートブックでは、特に次の 2 つの量を確認します。

- Lamb シフト (Lamb shift): 結合により各量子ビットの有効周波数がずれる効果
- 常在 ZZ 相互作用 (static ZZ interaction): 一方の量子ビットの状態に応じて他方の位相が変化する効果

Qubex の `QuantumSystem` では、各量子ビットを `Transmon`、量子ビット間の結合を `Coupling` として与えます。
そのうえで `get_static_zz`、`get_frequency_shift`、`get_effective_frequency` などのメソッドを用いて、結合による周波数シフトを確認できます。

In [1]:
from qubex.simulator import Control, Coupling, QuantumSimulator, QuantumSystem, Transmon

ここでは 2 つの三準位トランズモン `Q0` と `Q1` を用意し、その間に結合強度 0.01 GHz の `Coupling` を入れています。

- `strength=0.01`: 2 量子ビット間の結合強度

この設定により、相互作用を持つ 2 体系としてシミュレーションできます。

In [2]:
# Create the quantum system with two coupled transmon qubits (unit: GHz)

qubits = [
    Transmon(
        label="Q0",
        dimension=3,
        frequency=7.648,
        anharmonicity=-0.33,
        relaxation_rate=0.0,
        dephasing_rate=0.0,
    ),
    Transmon(
        label="Q1",
        dimension=3,
        frequency=8.275,
        anharmonicity=-0.33,
        relaxation_rate=0.0,
        dephasing_rate=0.0,
    ),
]

system = QuantumSystem(
    objects=qubits,
    couplings=[
        Coupling(
            pair=(qubits[0], qubits[1]),
            strength=0.01,
        ),
    ],
)

simulator = QuantumSimulator(system)

系全体のハミルトニアンを確認します。
`QuantumSystem.hamiltonian` は、各トランズモンのハミルトニアンと結合項を合わせた全ハミルトニアンを返します。

概念的には、全体のハミルトニアンは次のように書けます。

$$
H = H_0^{(Q0)} + H_0^{(Q1)} + H_{\mathrm{int}}
$$

ここで

$$
H_0^{(Qi)} = \omega_i a_i^\dagger a_i + \frac{\alpha_i}{2} a_i^\dagger a_i^\dagger a_i a_i
$$

であり、結合項 $H_{\mathrm{int}}$ によって 2 つのトランズモンが相互作用します。

Q0 の基底を $\ket{0}_{Q0},\ket{1}_{Q0},\ket{2}_{Q0}$ とし、Q1 の基底を $\ket{0}_{Q1},\ket{1}_{Q1},\ket{2}_{Q1}$ とし、系全体のハミルトニアンの基底は次の順番で定義することとします。

$$
\begin{align*}
\ket{0}_{Q0}\otimes\ket{0}_{Q1} \\
\ket{0}_{Q0}\otimes\ket{1}_{Q1} \\
\ket{0}_{Q0}\otimes\ket{2}_{Q1} \\
\ket{1}_{Q0}\otimes\ket{0}_{Q1} \\
\ket{1}_{Q0}\otimes\ket{1}_{Q1} \\
\ket{1}_{Q0}\otimes\ket{2}_{Q1} \\
\ket{2}_{Q0}\otimes\ket{0}_{Q1} \\
\ket{2}_{Q0}\otimes\ket{1}_{Q1} \\
\ket{2}_{Q0}\otimes\ket{2}_{Q1}
\end{align*}
$$

※以降 $\ket{i}_{Q0}\otimes\ket{j}_{Q1}$ は $\ket{ij}$ と書きます。

ここで対角項は、各基底状態 $\ket{ij}$ のエネルギーに対応します。
一方、非対角項は結合項 $H_{\mathrm{int}}$ に由来し、励起のやり取りによって異なる基底状態同士を結びます。

たとえば、$\ket{01}$ と $\ket{10}$ の間の非対角項は、片方の量子ビットの励起がもう片方へ移る過程に対応します。
同様に、$\ket{02}$ と $\ket{11}$、$\ket{11}$ と $\ket{20}$、$\ket{12}$ と $\ket{21}$ の間にも非対角項が現れます。

今回のように結合項が

$$
H_{\mathrm{int}} = g (a_0^\dagger a_1 + a_0 a_1^\dagger)
$$

の形をしていると、全励起数を保ったまま隣接する状態同士だけが結合します。
そのため、ハミルトニアン行列の非対角成分には、同じ全励起数を持つ状態同士の結合だけが現れます。

- 例1: $\ket{01}$ と $\ket{10}$
  - $0.01 \times (1 \times 1 + 0) \times 2 \pi \approx 0.063$
- 例2: $\ket{02}$ と $\ket{11}$
  - $0.01 \times (1 \times \sqrt{2} + 0) \times 2 \pi \approx 0.089$
- 例3: $\ket{12}$ と $\ket{21}$
  - $0.01 \times (\sqrt{2} \times \sqrt{2} + 0) \times 2 \pi \approx 0.126$

In [3]:
system.hamiltonian

Quantum object: dims=[[3, 3], [3, 3]], shape=(9, 9), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00]
 [0.00000000e+00 5.19933584e+01 0.00000000e+00 6.28318531e-02
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.01913266e+02 0.00000000e+00
  8.88576588e-02 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00]
 [0.00000000e+00 6.28318531e-02 0.00000000e+00 4.80538012e+01
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 8.88576588e-02 0.00000000e+00
  1.00047160e+02 0.00000000e+00 8.88576588e-02 0.00000000e+00
  0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 1.49967067e+02 0.00000000e+00 1.25663706e-01
  0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 

結合した 2 つのトランズモンのハミルトニアンを

$$
H = H_0 + H_{\mathrm{int}}
$$

とし、

$$
H_0 = \sum_{i=0,1} \left( \omega_i a_i^\dagger a_i + \frac{\alpha_i}{2} a_i^\dagger a_i^\dagger a_i a_i \right),
$$

$$
H_{\mathrm{int}} = g (a_0^\dagger a_1 + a_0 a_1^\dagger)
$$

とします。

分散領域では $g$ が量子ビット間の detuning $\Delta = \omega_0 - \omega_1$ や非調和度に比べて十分小さいため、結合項を摂動として扱えます。
このとき、励起の直接交換は抑えられますが、2 次の効果として各状態のエネルギーがわずかにずれます。

特に計算基底 $\{\ket{00}, \ket{01}, \ket{10}, \ket{11}\}$ に制限すると、実効ハミルトニアンは概ね

$$
H_{\mathrm{eff}} \approx -\frac{\tilde{\omega}_0}{2} ZI - \frac{\tilde{\omega}_1}{2} IZ + \frac{\zeta_{ZZ}}{4} ZZ
$$

の形になります。
ここで $\tilde{\omega}_0, \tilde{\omega}_1$ は Lamb シフトを含んだ有効周波数、$\zeta_{ZZ}$ は常在 ZZ 相互作用の強さです。

`QuantumSystem` の `get_static_zz`, `get_frequency_shift`, `get_effective_frequency` といった関数で常在 ZZ 相互作用の強さ、周波数シフト、実効周波数を確認できます。

In [4]:
static_zz = system.get_static_zz(("Q0", "Q1"))
print(f"static_zz: {static_zz * 1e6:.2f} kHz")

static_zz: -232.21 kHz


In [5]:
frequency_shift_0 = system.get_frequency_shift("Q0")
print(f"frequency_shift_0: {frequency_shift_0 * 1e6:.2f} kHz")

frequency_shift_1 = system.get_frequency_shift("Q1")
print(f"frequency_shift_1: {frequency_shift_1 * 1e6:.2f} kHz")

frequency_shift_0: -391.70 kHz
frequency_shift_1: -72.72 kHz


In [6]:
effective_frequency_0 = system.get_effective_frequency("Q0")
print(f"effective_frequency_0: {effective_frequency_0:.6f} GHz")

effective_frequency_1 = system.get_effective_frequency("Q1")
print(f"effective_frequency_1: {effective_frequency_1:.6f} GHz")

effective_frequency_0: 7.647608 GHz
effective_frequency_1: 8.274927 GHz


## 常在 ZZ 相互作用

常在 ZZ 相互作用による位相の蓄積を見やすくするため、ここでは両方の量子ビットに振幅 0 の `Control` を与えます。
これは「何も駆動しない」という意味ですが、各量子ビットの回転フレームを明示するために `frequency=effective_frequency_*` を指定しています。

また、相互作用の 1 周期に対応する時間として

$$
T = \left| \frac{1}{\text{static\_zz}} \right|
$$

を選んでいます。
この時間だけ自由発展させることで、ZZ 相互作用による位相回転を 1 周期ぶん観察できます。

In [7]:
# Duration for 1 period of the ZZ interaction
duration = abs(1 / static_zz)  # ns

controls = [
    Control(
        target=qubits[0],
        waveform=[0],
        durations=[duration],
        frequency=effective_frequency_0,
    ),
    Control(
        target=qubits[1],
        waveform=[0],
        durations=[duration],
        frequency=effective_frequency_1,
    ),
]

`Q0` を $\ket{0}$、`Q1` を $\ket{+}$ に初期化します。
$\ket{+}$ 状態は赤道面上の重ね合わせ状態なので、位相の蓄積が Bloch 球上の回転として見えやすくなります。

この場合、`Q0` が $\ket{0}$ に固定された条件で、`Q1` がどのように回転するかを確認できます。

In [8]:
result = simulator.mesolve(
    controls=controls,
    initial_state={
        qubits[0].label: "0",
        qubits[1].label: "+",
    },
    n_samples=101,
)

result.plot_bloch_vectors(qubits[0].label, frame="drive")
result.plot_bloch_vectors(qubits[1].label, frame="drive")
result.display_bloch_sphere(qubits[0].label, frame="drive")
result.display_bloch_sphere(qubits[1].label, frame="drive")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

次に、`Q0` を $\ket{1}$、`Q1` を $\ket{+}$ に初期化します。

直前の結果と比較すると、`Q0` の状態が $\ket{0}$ か $\ket{1}$ かによって、`Q1` の位相回転のしかたが変わることがわかります。
これが static ZZ interaction の本質であり、相手の量子ビットの状態に依存した位相シフトとして現れます。

In [9]:
result = simulator.mesolve(
    controls=controls,
    initial_state={
        qubits[0].label: "1",
        qubits[1].label: "+",
    },
    n_samples=101,
)

result.plot_bloch_vectors(qubits[0].label, frame="drive")
result.plot_bloch_vectors(qubits[1].label, frame="drive")
result.display_bloch_sphere(qubits[0].label, frame="drive")
result.display_bloch_sphere(qubits[1].label, frame="drive")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

最後に、両方の量子ビットを $\ket{+}$ に初期化します。

この場合は、両方の量子ビットが重ね合わせ状態にあるため、相互作用によって 2 体系全体の位相関係が時間発展します。
単一量子ビットの Bloch 球表示だけでは見えない多体系の相関が背後にありますが、各量子ビットの局所的な Bloch ベクトルの変化からも、結合による影響を確認できます。

In [10]:
result = simulator.mesolve(
    controls=controls,
    initial_state={
        qubits[0].label: "+",
        qubits[1].label: "+",
    },
    n_samples=101,
)

result.plot_bloch_vectors(qubits[0].label, frame="drive")
result.plot_bloch_vectors(qubits[1].label, frame="drive")
result.display_bloch_sphere(qubits[0].label, frame="drive")
result.display_bloch_sphere(qubits[1].label, frame="drive")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 有効ハミルトニアンの導出

Schrieffer-Wolff 変換
$$
\tilde H = e^{-S} H e^S
$$
を考え、反エルミート演算子 $S$ を
$$
[H_0, S] = -V
$$
を満たすように選びます。
このとき 2 次まで展開すると

$$
\tilde H \approx H_0 + \frac{1}{2}[V,S]
$$

となり、励起交換項 $V$ は消えて、エネルギー準位のずれだけが残ります。
このハミルトニアンを二準位系に近似することで有効ハミルトニアンが導出されます。

sympy を用いて Schrieffer-Wolff 変換を具体的に計算します。

以下のコードでは、 $S_{nm} = \frac{V_{nm}}{E_m - E_n}$ (にエルミート共役を減算したもの) とし、 $H_{\text{eff}} = H_0 + \frac{1}{2} [V, S]$ を計算しています。
$H_{\text{eff}}$ の非対角要素は $g^2$ のオーダーで非常に小さいことがわかります。

In [11]:
import sympy as sp
from sympy import Matrix, sqrt, symbols

# --- parameters ---
dim = 3
g, w0, w1, alpha0, alpha1 = symbols("g w0 w1 alpha0 alpha1", real=True)


# --- ladder operators ---
def annihilation(dim) -> Matrix:
    op = sp.zeros(dim)
    for n in range(1, dim):
        op[n - 1, n] = sqrt(n)
    return op


a = annihilation(dim)
adag = a.T
I = sp.eye(dim)


def kron(A, B) -> Matrix:
    return sp.kronecker_product(A, B)


# mode operators
a0 = kron(a, I)
a0dag = kron(adag, I)

a1 = kron(I, a)
a1dag = kron(I, adag)

n0 = a0dag * a0
n1 = a1dag * a1

# --- identity in full space ---
Id = sp.eye(dim * dim)

# --- H0 ---
H0 = (
    w0 * n0
    + w1 * n1
    + (alpha0 / 2) * (n0 * (n0 - Id))
    + (alpha1 / 2) * (n1 * (n1 - Id))
)

# --- V ---
V = g * (a0dag * a1 + a0 * a1dag)

# --- basis + energies ---
basis = []
energies = []

for i in range(dim):
    for j in range(dim):
        vec = sp.zeros(dim * dim, 1)
        idx = i * dim + j
        vec[idx] = 1

        basis.append((i, j, vec))

        E = w0 * i + alpha0 / 2 * i * (i - 1) + w1 * j + alpha1 / 2 * j * (j - 1)
        energies.append(E)

# --- construct S ---
S = sp.zeros(dim * dim)

for m, (_i1, _j1, vec_m) in enumerate(basis):
    for n, (_i2, _j2, vec_n) in enumerate(basis):
        if m == n:
            continue

        Vmn = (vec_m.T * V * vec_n)[0]

        if Vmn == 0:
            continue

        Em = energies[m]
        En = energies[n]

        S += (Vmn / (Em - En)) * (vec_m * vec_n.T)

# anti-Hermitian
S = -(S - S.T) / 2

# --- Heff ---
Heff = H0 + (V * S - S * V) / 2

sp.simplify(Heff)

Matrix([
[0,                                0,                                                                0,                               0,                                                                0,                                                                                        0,                                                                0,                                                                                       0,                             0],
[0, (-g**2 + w1*(w0 - w1))/(w0 - w1),                                                                0,                               0,                                                                0,                                                                                        0,                                                                0,                                                                                       0,                             0],
[0,                            

$$
[H_0, S] = -V
$$
を確かめます。

In [12]:
sp.simplify((H0 * S - S * H0) + V)

Matrix([
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [13]:
E00, E10, E01, E11 = Heff[0, 0], Heff[dim, dim], Heff[1, 1], Heff[dim + 1, dim + 1]

二準位系に近似します。

$$
H_{\text{eff}} = c_0 II + c_1 ZI + c_2 IZ + c_3 ZZ
$$

上のハミルトニアンではエネルギーは次のように計算できます。

$$
\begin{align*}
E_{00} &= c_0 + c_1 + c_2 + c_3 \\
E_{10} &= c_0 - c_1 + c_2 - c_3 \\
E_{01} &= c_0 + c_1 - c_2 - c_3 \\
E_{11} &= c_0 - c_1 - c_2 + c_3
\end{align*}
$$

このため、 $c_1,c_2,c_3$ は次のように求まります。

$$
\begin{align*}
c_1 &= \frac{E_{00} + E_{01} - E_{10} - E_{11}}{4} \\
c_2 &= \frac{E_{00} + E_{10} - E_{01} - E_{11}}{4} \\
c_3 &= \frac{E_{00} + E_{11} - E_{10} - E_{01}}{4}
\end{align*}
$$

$c_1 = \frac{-\omega_0^{\text{eff}}}{2}, c_2 = \frac{-\omega_1^{\text{eff}}}{2}, c_3 = \frac{\zeta_{ZZ}}{4}$ として、計算します。

In [14]:
c1 = sp.simplify(E00 + E01 - E10 - E11) / 4
c2 = sp.simplify(E00 + E10 - E01 - E11) / 4
c3 = sp.simplify(E00 + E11 - E10 - E01) / 4

In [15]:
zeta = 4 * c3
zeta

2*g**2*(-alpha0 - alpha1)/((alpha0 + w0 - w1)*(alpha1 - w0 + w1))

In [16]:
effective_omega0 = -2 * c1
effective_omega0

-g**2/(alpha1 - w0 + w1) - g**2/(alpha0 + w0 - w1) + g**2/(w0 - w1) + w0

In [17]:
effective_omega1 = -2 * c2
effective_omega1

-g**2/(alpha1 - w0 + w1) - g**2/(alpha0 + w0 - w1) - g**2/(w0 - w1) + w1

Qubex では $\zeta_{ZZ}/2$ を `static_zz` と呼んでいます。

In [18]:
static_zz = zeta / 2
static_zz

g**2*(-alpha0 - alpha1)/((alpha0 + w0 - w1)*(alpha1 - w0 + w1))

$\omega_i^{\text{eff}}$ から $\omega_i$ と `static_zz` を引くことで Lamb シフトの量が計算できます。

In [19]:
lamb0 = sp.simplify(effective_omega0 - w0 - static_zz)
lamb0

g**2/(w0 - w1)

In [20]:
lamb1 = sp.simplify(effective_omega1 - w1 - static_zz)
lamb1

-g**2/(w0 - w1)

以上の背景から `QuantumSystem` では次のように実装しています。

```python
    def get_effective_frequency(self, label: str) -> float:
        """Return the effective frequency including shifts."""
        obj = self.get_object(label)
        shift = self.get_frequency_shift(label)
        return obj.frequency + shift

    def get_frequency_shift(self, label: str) -> float:
        """Return total frequency shift from couplings."""
        shift = 0.0
        for neighbor in self.graph.neighbors(label):
            shift += self.get_lamb_shift((label, neighbor))
            shift += self.get_static_zz((label, neighbor))
        return shift

    def get_lamb_shift(self, label: str | tuple[str, str]) -> float:
        """Return the Lamb shift for a coupling pair."""
        pair = self.to_tuple_pair(label)
        coupling = self.get_coupling(pair)
        obj_0 = self.get_object(pair[0])
        obj_1 = self.get_object(pair[1])

        g = coupling.strength
        delta = obj_0.frequency - obj_1.frequency
        return (g**2) / delta

    def get_static_zz(self, label: str | tuple[str, str]) -> float:
        """Return the static ZZ shift for a coupling pair."""
        pair = self.to_tuple_pair(label)
        obj_0 = self.get_object(pair[0])
        obj_1 = self.get_object(pair[1])

        if obj_0.dimension < 3 or obj_1.dimension < 3:
            return 0.0

        g = self.get_coupling(pair).strength
        delta = obj_0.frequency - obj_1.frequency
        alpha_0 = obj_0.anharmonicity
        alpha_1 = obj_1.anharmonicity
        xi = g**2 * (alpha_0 + alpha_1) / ((delta + alpha_0) * (delta - alpha_1))
        return xi
```